# 🔮 Notebook 05 — Scenario Analysis
> **Market Demand Trend Analysis** | *Best Case / Worst Case / Most Likely*

**Objectives**
- Define three market scenarios with documented assumptions
- Build scenario-specific demand projections using Prophet
- Identify which roles and skills are most/least resilient per scenario
- Create an interactive Plotly dashboard for scenario exploration
- Output a scenario comparison table

---

### Scenario Definitions

| Scenario | Demand Multiplier | Assumption |
|---|---|---|
| 🟢 Best Case | +30% uplift | AI adoption accelerates; high outsourcing demand |
| 🟡 Most Likely | baseline | Current trend continues linearly |
| 🔴 Worst Case | −25% reduction | Economic slowdown; hiring freeze in tech |

---

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from prophet import Prophet

from utils import (
    set_style, load_merged, add_role_category, build_daily_series, explode_skills,
    savefig, save_plotly, PALETTE, PLOTLY_TEMPLATE, OUT_REP
)

set_style()
print('✅ Setup complete')

## 1. Load Data & Build Base Series

In [ ]:
clean_path = OUT_REP / 'merged_clean.parquet'
if clean_path.exists():
    df = pd.read_parquet(clean_path)
else:
    df = load_merged()
    df = add_role_category(df)

daily = build_daily_series(df, date_col='first_seen')
print(f'Base series length: {len(daily)} days')
print(daily)

## 2. Define Scenario Parameters

In [ ]:
SCENARIOS = {
    'Best Case': {
        'multiplier'  : 1.30,
        'color'       : '#2ECC71',
        'color_fill'  : 'rgba(46,204,113,0.15)',
        'description' : 'AI adoption accelerates globally; demand for data roles surges by 30%. '
                        'Outsourcing contracts increase. Remote work normalised.',
        'assumptions' : [
            'GDP growth > 2.5%',
            'Tech sector hiring increases',
            'GenAI adoption drives new role creation',
            'Outsourcing demand +30%',
        ],
    },
    'Most Likely': {
        'multiplier'  : 1.00,
        'color'       : '#F5A623',
        'color_fill'  : 'rgba(245,166,35,0.15)',
        'description' : 'Current trends continue. Moderate growth in data/ML roles. '
                        'Stable outsourcing market.',
        'assumptions' : [
            'GDP growth 1–2%',
            'Tech layoffs stabilise',
            'Gradual AI adoption',
            'Outsourcing demand stable',
        ],
    },
    'Worst Case': {
        'multiplier'  : 0.75,
        'color'       : '#E74C3C',
        'color_fill'  : 'rgba(231,76,60,0.15)',
        'description' : 'Economic slowdown and hiring freeze in tech. 25% reduction in demand. '
                        'Companies consolidate outsourcing vendors.',
        'assumptions' : [
            'GDP contraction or near-zero growth',
            'Major tech sector layoffs',
            'Budget cuts freeze new hires',
            'Outsourcing spend cut by 25%',
        ],
    },
}

print('Scenario parameters defined:')
for name, params in SCENARIOS.items():
    print(f'  {name}: ×{params["multiplier"]}')

## 3. Generate Scenario Forecasts (30-Day Horizon)

In [ ]:
FORECAST_DAYS = 30

# Fit base Prophet model
prophet_df = daily.reset_index()
prophet_df.columns = ['ds', 'y']
prophet_df['ds'] = pd.to_datetime(prophet_df['ds'])

base_model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=False,
    changepoint_prior_scale=0.3,
    interval_width=0.80
)
base_model.fit(prophet_df)

future = base_model.make_future_dataframe(periods=FORECAST_DAYS)
base_fc = base_model.predict(future)

# Apply multiplier per scenario
scenario_forecasts = {}
for name, params in SCENARIOS.items():
    fc = base_fc[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    fc['yhat']       = (fc['yhat']       * params['multiplier']).clip(lower=0)
    fc['yhat_lower'] = (fc['yhat_lower'] * params['multiplier']).clip(lower=0)
    fc['yhat_upper'] = (fc['yhat_upper'] * params['multiplier']).clip(lower=0)
    scenario_forecasts[name] = fc

print(f'Scenarios generated — {FORECAST_DAYS}-day horizon')

## 4. Scenario Comparison Chart (Interactive)

In [ ]:
fig = go.Figure()

# Actual data
fig.add_trace(go.Scatter(
    x=daily.index, y=daily.values,
    name='Actual Data', mode='lines+markers',
    line=dict(color=PALETTE['primary'], width=3),
    marker=dict(size=8)
))

# Forecast start line
fig.add_vline(
    x=str(daily.index[-1]),
    line_dash='dash', line_color='#888888',
    annotation_text='Forecast →',
    annotation_position='top right'
)

# Scenarios
for name, params in SCENARIOS.items():
    fc   = scenario_forecasts[name]
    fcst = fc[fc['ds'] > daily.index[-1]]  # future only

    fig.add_trace(go.Scatter(
        x=fcst['ds'], y=fcst['yhat'],
        name=f'{name} (×{params["multiplier"]})',
        mode='lines',
        line=dict(color=params['color'], width=2.5, dash='dot')
    ))

    fig.add_trace(go.Scatter(
        x=list(fcst['ds']) + list(fcst['ds'][::-1]),
        y=list(fcst['yhat_upper']) + list(fcst['yhat_lower'][::-1]),
        fill='toself',
        fillcolor=params['color_fill'],
        line=dict(color='rgba(0,0,0,0)'),
        name=f'{name} CI',
        showlegend=False
    ))

fig.update_layout(
    title=f'Job Market Demand — 3-Scenario Forecast ({FORECAST_DAYS} days)',
    xaxis_title='Date',
    yaxis_title='Daily Job Postings',
    template=PLOTLY_TEMPLATE,
    height=550,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
save_plotly(fig, '05_scenario_forecast')
fig.show()

## 5. Scenario Assumptions Dashboard

In [ ]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=list(SCENARIOS.keys()),
    horizontal_spacing=0.06
)

for col_idx, (name, params) in enumerate(SCENARIOS.items(), start=1):
    fc   = scenario_forecasts[name]
    fcst = fc[fc['ds'] > daily.index[-1]]

    # Actual
    fig.add_trace(go.Scatter(
        x=daily.index, y=daily.values,
        name='Actual', showlegend=(col_idx == 1),
        line=dict(color=PALETTE['primary'], width=2),
        mode='lines+markers', marker=dict(size=5)
    ), row=1, col=col_idx)

    # Scenario forecast
    fig.add_trace(go.Scatter(
        x=fcst['ds'], y=fcst['yhat'],
        name=name, showlegend=False,
        line=dict(color=params['color'], width=2.5)
    ), row=1, col=col_idx)

    # CI band
    fig.add_trace(go.Scatter(
        x=list(fcst['ds']) + list(fcst['ds'][::-1]),
        y=list(fcst['yhat_upper']) + list(fcst['yhat_lower'][::-1]),
        fill='toself', fillcolor=params['color_fill'],
        line=dict(color='rgba(0,0,0,0)'),
        showlegend=False
    ), row=1, col=col_idx)

fig.update_layout(
    title='Side-by-Side Scenario View',
    template=PLOTLY_TEMPLATE, height=420
)
save_plotly(fig, '05_scenario_side_by_side')
fig.show()

## 6. Role Resilience per Scenario

In [ ]:
role_counts = df['role_category'].value_counts().reset_index()
role_counts.columns = ['role', 'baseline_postings']

for name, params in SCENARIOS.items():
    role_counts[name] = (role_counts['baseline_postings'] * params['multiplier']).round().astype(int)

print('Role demand under each scenario:')
role_counts.set_index('role')

In [ ]:
scenario_cols = list(SCENARIOS.keys())
colors_sc     = [SCENARIOS[s]['color'] for s in scenario_cols]

fig = go.Figure()
for sc, color in zip(scenario_cols, colors_sc):
    fig.add_trace(go.Bar(
        name=sc,
        x=role_counts['role'],
        y=role_counts[sc],
        marker_color=color,
        opacity=0.85
    ))

fig.update_layout(
    barmode='group',
    title='Projected Role Demand under Each Scenario',
    xaxis_title='Role Category',
    yaxis_title='Estimated Postings',
    template=PLOTLY_TEMPLATE,
    xaxis_tickangle=-30,
    height=480,
    legend=dict(orientation='h', y=1.1, x=0)
)
save_plotly(fig, '05_role_demand_by_scenario')
fig.show()

## 7. Scenario Summary Table

In [ ]:
summary_rows = []
for name, params in SCENARIOS.items():
    fc   = scenario_forecasts[name]
    fcst = fc[fc['ds'] > daily.index[-1]]
    summary_rows.append({
        'Scenario'            : name,
        'Multiplier'          : params['multiplier'],
        'Mean Daily Demand'   : round(fcst['yhat'].mean(), 1),
        'Peak Daily Demand'   : round(fcst['yhat'].max(), 1),
        'Min Daily Demand'    : round(fcst['yhat'].min(), 1),
        'Total (30d)'         : round(fcst['yhat'].sum()),
        'Description'         : params['description'][:80] + '...'
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.set_index('Scenario', inplace=True)
print(summary_df.to_string())

# Save to CSV
summary_df.to_csv(OUT_REP / '05_scenario_summary.csv')
print('\n✅ Saved scenario summary → outputs/reports/05_scenario_summary.csv')

## 8. Skills at Risk / Growing per Scenario

In [ ]:
skills_exploded = explode_skills(df, skill_col='job_skills')
top_skills = skills_exploded['skill'].value_counts().head(15).reset_index()
top_skills.columns = ['skill', 'baseline']

for name, params in SCENARIOS.items():
    top_skills[name] = (top_skills['baseline'] * params['multiplier']).round().astype(int)

fig = px.bar(
    top_skills.melt(id_vars='skill', value_vars=scenario_cols,
                    var_name='Scenario', value_name='Demand'),
    x='skill', y='Demand', color='Scenario', barmode='group',
    title='Top Skills Demand Projection by Scenario',
    template=PLOTLY_TEMPLATE,
    color_discrete_map={
        'Best Case'   : SCENARIOS['Best Case']['color'],
        'Most Likely' : SCENARIOS['Most Likely']['color'],
        'Worst Case'  : SCENARIOS['Worst Case']['color'],
    }
)
fig.update_layout(xaxis_tickangle=-35, height=480)
save_plotly(fig, '05_skills_scenario_projection')
fig.show()

---
## Scenario Summary

| Scenario | Multiplier | 30-day Outlook | Strategic Action |
|---|---|---|---|
| 🟢 Best Case | ×1.30 | +30% demand surge | Scale up Data Engineer, ML Engineer pipelines |
| 🟡 Most Likely | ×1.00 | Stable growth | Maintain current recruitment focus |
| 🔴 Worst Case | ×0.75 | −25% contraction | Pivot to high-value niches (AI, Security) |

➡️ **Next: Notebook 06 — Model Validation**